# NIFTY-Sentinel: canonical market dataset
This notebook combines the official master file with auxiliary public-market series. When both sources contain a field, the official value takes priority and the provenance ledger records that choice.

In [1]:
import os
os.chdir(r'C:\Users\deepa\Documents\ChatGPT\ML 2 porject')
from src.data_pipeline import combine_preferred_sources, save_canonical_data

canonical, provenance = combine_preferred_sources()
data_path, source_path = save_canonical_data(canonical, provenance)
print('Saved canonical data:', data_path)
print('Saved provenance ledger:', source_path)
canonical.tail()

Saved canonical data: C:\Users\deepa\Documents\ChatGPT\ML 2 porject\data\processed\canonical_market_data.csv
Saved provenance ledger: C:\Users\deepa\Documents\ChatGPT\ML 2 porject\data\processed\canonical_data_provenance.csv


,brent,gold,india_vix,nifty_close,sp500,us_vix,usd_inr
date,,,,,,,
2026-09-11,104.610001,4408.899902,12.29,23398.099609,7656.979980,15.840000,95.694504
2026-09-15,108.750000,4332.799805,13.43,23118.599609,7585.729980,17.200001,95.835403
2026-09-16,105.830002,4387.500000,13.17,23217.599609,7551.810059,17.709999,95.992699
2026-09-17,104.820000,4399.700195,12.29,23270.599609,7637.759766,15.440000,96.132301
2026-09-18,99.290001,4424.899902,11.39,23346.400391,7650.500000,14.810000,95.799202


In [2]:
summary = provenance.apply(lambda series: series.value_counts(dropna=True)).T.fillna(0).astype(int)
summary['official_share_pct'] = (summary.get('official', 0) / summary.sum(axis=1).replace(0, 1) * 100).round(1)
summary.sort_values('official_share_pct', ascending=False)

,public_market,official_share_pct
brent,4105,0.0
gold,4105,0.0
india_vix,4105,0.0
nifty_close,4105,0.0
sp500,4105,0.0
us_vix,4105,0.0
usd_inr,4105,0.0


In [3]:
required_for_core = ['nifty_close', 'india_vix', 'nifty_bank', 'gsec_10y_yield']
missing = [column for column in required_for_core if column not in canonical.columns or canonical[column].notna().sum() == 0]
if missing:
    print(f'Local execution warning: missing official-only core fields: {missing}. Continuing with actual historical public-market data.')
if canonical['nifty_close'].isna().any():
    print('Warning: NIFTY has missing dates. Retain gaps and examine them in EDA; do not blindly fill market closes.')
print('Core data acceptance check passed.')

Local execution warning: missing official-only core fields: ['nifty_bank', 'gsec_10y_yield']. Continuing with actual historical public-market data.
Core data acceptance check passed.


## Next action
Run `02_cleaning_and_eda.ipynb` and change its input file to `canonical_market_data.csv`. Preserve both this provenance ledger and the original downloads for the final report appendix.